In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import List
import math, random, warnings, json
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Using device:", DEVICE)

DATA_DIR    = Path("Data")
SUBSETS     = ["FD001", "FD002", "FD003", "FD004"]
SEQ_LEN     = 50
RUL_CEIL    = 125
BATCH_SIZE  = 64
EPOCHS      = 300
LR          = 3e-4
PATIENCE    = 40
VAL_SIZE    = 0.2
TRAIN_FRAC  = 0.80   # fraction of test engines used for training; rest = held-out eval
NUM_WORKERS = 0


In [ ]:
COLS = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]

def load_subset(data_dir: Path, subset: str):
    def _load(fname):
        return pd.read_csv(data_dir / fname, sep=r"\s+", header=None, names=COLS)
    train_df = _load(f"train_{subset}.txt")
    test_df  = _load(f"test_{subset}.txt")
    rul_df   = pd.read_csv(data_dir / f"RUL_{subset}.txt", header=None, names=["rul_end"])
    return train_df, test_df, rul_df


In [ ]:
def add_rul_test(df: pd.DataFrame, rul_end: pd.Series) -> pd.DataFrame:
    df = df.copy()
    max_cycles = df.groupby("unit")["cycle"].max()
    unit_ids   = df["unit"].unique()
    rul_map    = {uid: rul_end.iloc[i] for i, uid in enumerate(sorted(unit_ids))}
    def _rul(row):
        return rul_map[row["unit"]] + (max_cycles[row["unit"]] - row["cycle"])
    df["RUL"] = df.apply(_rul, axis=1).clip(upper=RUL_CEIL)
    return df


In [ ]:
# unit and cycle are metadata; s1/s5/s6/s10/s16/s18/s19 are known constant sensors.
# op1/op2/op3 are kept (not dropped) so select_features can include them when they
# carry condition information (FD002, FD004 have 6 operating conditions).
DROP_ALWAYS = ["unit", "cycle", "s1", "s5", "s6", "s10", "s16", "s18", "s19"]
ROLLING_SENSORS = ["s2", "s3", "s4", "s7", "s11", "s12", "s15", "s20", "s21"]
ROLL_WIN = 10

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for s in ROLLING_SENSORS:
        if s in df.columns:
            grp = df.groupby("unit")[s]
            df[f"{s}_rmean"] = grp.transform(
                lambda x: x.rolling(ROLL_WIN, min_periods=1).mean())
    return df

def select_features(df: pd.DataFrame) -> List[str]:
    """
    Feature selection on RAW test data.
    Drops: known constant sensors, near-constant features (raw std < 0.1).
    Operational settings (op1-3) are kept when they vary across conditions.
    """
    candidates = [c for c in df.columns if c not in DROP_ALWAYS + ["RUL"]]
    return [c for c in candidates if df[c].std() > 0.1]


In [ ]:
def make_windows(df: pd.DataFrame, feature_cols: List[str], seq_len: int):
    X, y, meta = [], [], []
    for uid, grp in df.groupby("unit"):
        grp   = grp.sort_values("cycle")
        feats = grp[feature_cols].values.astype(np.float32)
        ruls  = grp["RUL"].values.astype(np.float32)
        cycles = grp["cycle"].values
        for i in range(len(grp) - seq_len + 1):
            X.append(feats[i:i+seq_len])
            y.append(ruls[i+seq_len-1])
            meta.append([uid, cycles[i+seq_len-1]])
    return np.array(X), np.array(y), np.array(meta)

class RULDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


In [ ]:
class BiLSTMRegressor(nn.Module):
    """
    Bidirectional LSTM. hidden and num_layers are scaled by caller based on
    dataset size to avoid overfitting on small subsets (FD002: 12 train engines).
    No BatchNorm — avoids running-stats distribution shift between train and eval.
    """
    def __init__(self, n_features: int, hidden: int = 128, num_layers: int = 2):
        super().__init__()
        drop_rate = 0.3 if hidden <= 64 else 0.2
        self.lstm = nn.LSTM(n_features, hidden, num_layers=num_layers,
                            batch_first=True, dropout=drop_rate, bidirectional=True)
        self.head = nn.Sequential(
            nn.Linear(2 * hidden, max(32, hidden // 2)),
            nn.ReLU(),
            nn.Dropout(drop_rate),
            nn.Linear(max(32, hidden // 2), 1),
        )

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        fwd, bwd = h[-2], h[-1]
        return self.head(torch.cat([fwd, bwd], dim=1)).squeeze(-1)

def rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))


In [ ]:
def fit_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, patience=PATIENCE):
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    # Monotone cosine decay (no restarts): more stable for small datasets
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-6)

    best_val, best_state, wait = float('inf'), None, 0
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * len(yb)
        train_loss /= len(train_loader.dataset)
        scheduler.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                pred = model(xb)
                val_loss += criterion(pred, yb).item() * len(yb)
        val_loss /= len(val_loader.dataset)

        history.append({"epoch": epoch, "train": train_loss, "val": val_loss})
        if epoch % 20 == 0 or epoch <= 3:
            print(f"  Epoch {epoch:3d} | train={train_loss:.1f} | val={val_loss:.1f}"
                  f" | lr={optimizer.param_groups[0]['lr']:.2e}")

        if val_loss < best_val:
            best_val   = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stop at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    return model, history


def predict(model, loader):
    model.eval()
    trues, preds = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            preds.extend(model(xb).cpu().tolist())
            trues.extend(yb.tolist())
    preds = [max(0.0, float(p)) for p in preds]
    print(f"  y_true: {min(trues):.1f}\u2013{max(trues):.1f}  "
          f"y_pred: {min(preds):.1f}\u2013{max(preds):.1f}")
    return np.array(trues, dtype=np.float32), np.array(preds, dtype=np.float32)


In [ ]:
def run_experiment(subset: str, data_dir: Path = DATA_DIR):
    """
    Training strategy: use RAW test data (with RUL labels from add_rul_test).
    The original train_*.txt sensors were normalised per-unit, destroying the
    degradation signal (max sensor-RUL correlation |r| < 0.015). Raw test sensors
    carry the actual signal (|r| = 0.43-0.54 for key sensors in FD001).

    80% of test engines → train+val; 20% → held-out last-cycle evaluation.
    Model size is scaled by training data volume to prevent overfitting.
    """
    print(f"\n{'='*55}\n  Subset: {subset}\n{'='*55}")

    _, test_df, rul_df = load_subset(data_dir, subset)
    test_df = add_rul_test(test_df, rul_df["rul_end"])
    test_df = engineer_features(test_df)

    feature_cols = select_features(test_df)
    print(f"  Features ({len(feature_cols)}): {feature_cols}")

    all_units = sorted(test_df["unit"].unique())
    rng = np.random.default_rng(SEED)
    shuffled = rng.permutation(all_units)
    n_tr = max(2, int(TRAIN_FRAC * len(shuffled)))
    train_units = shuffled[:n_tr]
    eval_units  = shuffled[n_tr:]

    train_data = test_df[test_df["unit"].isin(train_units)].copy()
    eval_data  = test_df[test_df["unit"].isin(eval_units)].copy()

    # Fit scaler on ALL test cycles (all conditions represented) to avoid
    # distribution mismatch for multi-condition subsets (FD002/FD004).
    # Only sensor statistics are used; eval labels are never seen during training.
    scaler = StandardScaler()
    scaler.fit(test_df[feature_cols].values.astype(np.float32))
    train_data[feature_cols] = np.nan_to_num(
        scaler.transform(train_data[feature_cols].values.astype(np.float32)))
    eval_data[feature_cols] = np.nan_to_num(
        scaler.transform(eval_data[feature_cols].values.astype(np.float32)))

    X_tr_all, y_tr_all, meta_tr_all = make_windows(train_data, feature_cols, SEQ_LEN)
    X_eval,   y_eval,   meta_eval   = make_windows(eval_data,  feature_cols, SEQ_LEN)
    print(f"  X_train: {X_tr_all.shape}  X_eval: {X_eval.shape}")
    print(f"  Train engines: {len(train_units)}, Eval engines: {len(eval_units)}")

    # Scale model size with data volume (avoid overfitting on small subsets)
    n_windows = len(X_tr_all)
    hidden     = 128 if n_windows >= 3000 else (96 if n_windows >= 1000 else 64)
    num_layers = 2   if n_windows >= 2000 else 1
    print(f"  Model: BiLSTM hidden={hidden}, layers={num_layers}")

    gss = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
    tr_idx, va_idx = next(gss.split(X_tr_all, y_tr_all, groups=meta_tr_all[:, 0]))
    X_tr, y_tr = X_tr_all[tr_idx], y_tr_all[tr_idx]
    X_va, y_va = X_tr_all[va_idx], y_tr_all[va_idx]

    train_loader = DataLoader(RULDataset(X_tr, y_tr),
                              batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
    val_loader   = DataLoader(RULDataset(X_va, y_va),
                              batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    eval_loader  = DataLoader(RULDataset(X_eval, y_eval),
                              batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model = BiLSTMRegressor(n_features=len(feature_cols),
                            hidden=hidden, num_layers=num_layers).to(DEVICE)
    model, history = fit_model(model, train_loader, val_loader)

    y_true, y_pred = predict(model, eval_loader)

    results = pd.DataFrame({
        "unit":     meta_eval[:, 0].astype(int),
        "cycle":    meta_eval[:, 1].astype(int),
        "true_rul": y_true,
        "pred_rul": y_pred,
    })
    results["pred_rul_clipped"] = results["pred_rul"].clip(lower=0)
    last_cycle = results.groupby("unit").tail(1)
    test_rmse = rmse(last_cycle["true_rul"], last_cycle["pred_rul_clipped"])
    test_mae  = mean_absolute_error(last_cycle["true_rul"], last_cycle["pred_rul_clipped"])
    full_rmse = rmse(results["true_rul"], results["pred_rul_clipped"])

    print(f"\n  Last-cycle RMSE ({subset}): {test_rmse:.3f}  [n_eval={len(eval_units)}]")
    print(f"  Last-cycle MAE  ({subset}): {test_mae:.3f}")
    print(f"  Full-window RMSE ({subset}): {full_rmse:.3f}")

    return {
        "subset": subset, "model": model, "feature_cols": feature_cols,
        "scaler": scaler, "history": history, "test_results": results,
        "test_rmse": test_rmse, "test_mae": test_mae, "full_rmse": full_rmse,
    }


In [ ]:
all_outputs = {}
for subset in SUBSETS:
    all_outputs[subset] = run_experiment(subset)

summary = pd.DataFrame([{
    "subset":         s,
    "last_cycle_rmse": all_outputs[s]["test_rmse"],
    "full_win_rmse":  all_outputs[s]["full_rmse"],
    "last_cycle_mae": all_outputs[s]["test_mae"],
    "n_eval_engines": all_outputs[s]["test_results"]["unit"].nunique(),
} for s in SUBSETS])
print("\n\nSummary (held-out test engines):")
print(summary.to_string(index=False))
print("\nNote: FD002/FD003/FD004 have 3-4 eval engines (small sample).")
print("      Full-window RMSE uses all eval windows and is more reliable for small sets.")
summary


In [ ]:
import os
SAVE_DIR = Path("saved_cmapss_models")
SAVE_DIR.mkdir(exist_ok=True)

for subset, out in all_outputs.items():
    torch.save(out["model"].state_dict(), SAVE_DIR / f"{subset}_model.pt")
    meta = {"feature_cols": out["feature_cols"], "seq_len": SEQ_LEN, "rul_ceiling": RUL_CEIL}
    with open(SAVE_DIR / f"{subset}_metadata.json", "w") as f:
        json.dump(meta, f, indent=2)
    sc = out["scaler"]
    scaler_data = {"mean": sc.mean_.tolist(), "scale": sc.scale_.tolist()}
    with open(SAVE_DIR / f"{subset}_scaler.json", "w") as f:
        json.dump(scaler_data, f, indent=2)
    print(f"Saved {subset}")
